# Example 4 - Exact cylinder h-transform in d = 2: blob to annulus

This notebook builds a **stable exact cylinder example** in the sense of the paper's Doob h-transform construction.

Instead of the nonlinear moment target from Algorithm 2, we use the positive cylinder terminal datum

$$
g(\mu) = \exp\bigl(c\,\mu(\phi)\bigr),
$$

where $\phi$ is a smooth periodic annulus observable on $\mathbb{T}^2$. This stays inside the same h-transform framework, but avoids the oscillatory $\eta$-quadrature that made the nonlinear cylinder examples fragile in practice.

The visual setup is:

- 192 particles with equal frozen masses $s_i = 1/n$
- a compact initial blob near the center of the torus
- a smooth annulus observable $\phi$ centered at the same point
- a comparison between the free diffusion and the exact h-transformed diffusion

The conditioned process opens a clear hole and forms a ring, while the free process remains a blurred blob.


## Exact factorization for this cylinder terminal datum

For fixed masses $s = (s_1,\dots,s_n)$ and particle locations $x = (x_1,\dots,x_n)$, the finite-dimensional terminal datum is

$$
g_n(s,x)
= \exp\left(c \sum_{i=1}^n s_i \phi(x_i)\right)
= \prod_{i=1}^n \exp\left(c s_i \phi(x_i)\right).
$$

Because the free particle system is independent across labels, the backward heat solution factorizes exactly:

$$
u_t^{(n)}(s,x) = \prod_{i=1}^n \theta_{i,t}(x_i),
$$

with

$$
\theta_{i,t}(z) = P_{\mathbb{T}^2, (T-t)/s_i}\!\left[e^{c s_i \phi(\cdot)}\right](z).
$$

Hence the h-transform drift is

$$
b_i(t,x;s)
= 2 D_i^{(n)} \log u_t^{(n)}(s,x)
= \frac{2}{s_i} \, \nabla \log \theta_{i,t}(x_i).
$$

This is still an exact h-transform example from the paper, but numerically it is much cleaner:

- no Monte Carlo
- no Sinkhorn
- no $\eta$-quadrature
- only one-particle heat semigroups evaluated spectrally on the torus

The time discretization is still Euler-Maruyama, just as in the other numerical examples.


In [ ]:
import numpy as np
import pandas as pd
import sys
from itertools import product
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots


ROOT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents, Path("/mnt/data").resolve())
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "wasserstein_conditioning_algorithms.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate wasserstein_conditioning_algorithms.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from notebooks.support import (
    circle_trace,
    configure_plotly,
    make_seed_grid_figure,
    make_two_panel_particle_animation,
)
from wasserstein_conditioning_algorithms import (
    ParticleSimulation,
    shortest_periodic_displacement,
    wrap_torus,
)

configure_plotly()
np.set_printoptions(precision=4, suppress=True)


In [ ]:
# --- Plotting helpers shared across the example notebooks ---


def make_comparison_animation(
    *,
    free_positions,
    conditioned_positions,
    times,
    color_values,
    hover_text,
    static_traces_left=None,
    static_traces_right=None,
    title,
    marker_size=8,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
):
    return make_two_panel_particle_animation(
        left_positions=free_positions,
        right_positions=conditioned_positions,
        times=times,
        color_values=color_values,
        hover_text=hover_text,
        title=title,
        left_static_traces=static_traces_left,
        right_static_traces=static_traces_right,
        left_subplot_title="Free diffusion",
        right_subplot_title="Exact cylinder h-transform",
        left_name="free particles",
        right_name="conditioned particles",
        marker_size=marker_size,
        x_range=x_range,
        y_range=y_range,
        colorscale="HSV",
        cmin=0.0,
        cmax=1.0,
        colorbar_title="initial angle",
        time_formatter=lambda t: f"time = {t:.8f}",
    )


In [ ]:
# --- Exact factorized cylinder simulation helpers ---

def sunflower_disk(n, *, center=(0.5, 0.5), radius=0.055):
    center = np.asarray(center, dtype=np.float64)
    k = np.arange(n, dtype=np.float64)
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))
    r = radius * np.sqrt((k + 0.5) / n)
    theta = golden_angle * k
    pts = np.stack(
        [
            center[0] + r * np.cos(theta),
            center[1] + r * np.sin(theta),
        ],
        axis=1,
    )
    return wrap_torus(pts)


def periodic_annulus_observable(*, center=(0.5, 0.5), radius=0.16, sigma=0.045, image_radius=1):
    center = np.asarray(center, dtype=np.float64)
    shifts = np.array(list(product(range(-image_radius, image_radius + 1), repeat=2)), dtype=np.float64)

    def phi(points):
        pts = np.asarray(points, dtype=np.float64)
        if pts.ndim == 1:
            pts = pts[None, :]
        diffs = pts[:, None, :] - center[None, None, :] + shifts[None, :, :]
        r = np.linalg.norm(diffs, axis=-1)
        values = np.exp(-0.5 * ((r - radius) / sigma) ** 2).sum(axis=1)
        return values

    return phi


class ScalarTorusHeatSemigroup:
    """Spectral evaluator for P_t[e^{scale * phi}] and its gradient on T^2."""

    def __init__(self, scalar_function, *, dimension=2, grid_shape=32):
        if isinstance(grid_shape, int):
            grid_shape = (grid_shape,) * dimension
        self.dimension = int(dimension)
        self.grid_shape = tuple(int(g) for g in grid_shape)

        axes = [np.arange(g, dtype=np.float64) / g for g in self.grid_shape]
        mesh = np.meshgrid(*axes, indexing="ij")
        self.grid_points = np.stack([a.ravel() for a in mesh], axis=-1)

        samples = np.asarray(scalar_function(self.grid_points), dtype=np.float64)
        self.scalar_values = samples.reshape(self.grid_shape)
        self.num_grid_points = int(np.prod(self.grid_shape))

        freq_axes = [np.fft.fftfreq(g, d=1.0 / g).astype(np.float64) for g in self.grid_shape]
        freq_mesh = np.meshgrid(*freq_axes, indexing="ij")
        self.k_vectors = np.stack([a.ravel() for a in freq_mesh], axis=-1)
        self.k_sq_norm = np.sum(self.k_vectors ** 2, axis=1)

        self._coeff_cache = {}

    def _coefficients(self, scale):
        key = round(float(scale), 14)
        if key not in self._coeff_cache:
            sample_values = np.exp(scale * self.scalar_values)
            coeffs = np.fft.fftn(sample_values).reshape(-1) / self.num_grid_points
            self._coeff_cache[key] = coeffs
        return self._coeff_cache[key]

    def evaluate(self, scale, time, x):
        coeffs = self._coefficients(scale)
        point = wrap_torus(np.asarray(x, dtype=np.float64).reshape(self.dimension))

        decay = np.exp(-4.0 * np.pi ** 2 * self.k_sq_norm * time)
        phase = np.exp(2j * np.pi * (self.k_vectors @ point))
        terms = coeffs * decay * phase

        value = float(np.real(np.sum(terms)))
        gradient = np.real(np.sum((2j * np.pi * self.k_vectors) * terms[:, None], axis=0))
        return value, gradient


def simulate_free_diffusion(*, masses, horizon, step_size, initial_positions, rng):
    masses = np.asarray(masses, dtype=np.float64)
    x = wrap_torus(np.asarray(initial_positions, dtype=np.float64))
    m_steps = int(round(horizon / step_size))
    times = np.linspace(0.0, horizon, m_steps + 1)
    n, d = x.shape

    positions = np.empty((m_steps + 1, n, d), dtype=np.float64)
    positions[0] = x
    noise_scale = np.sqrt(2.0 * step_size / masses)[:, None]

    for m in range(m_steps):
        x = wrap_torus(x + noise_scale * rng.normal(size=(n, d)))
        positions[m + 1] = x

    return ParticleSimulation(times=times, positions=positions, masses=masses)


def simulate_exact_cylinder_em(
    *,
    masses,
    observable,
    c,
    horizon,
    step_size,
    initial_positions,
    grid_shape=32,
    rng,
    store_drifts=True,
):
    masses = np.asarray(masses, dtype=np.float64)
    x = wrap_torus(np.asarray(initial_positions, dtype=np.float64))
    m_steps = int(round(horizon / step_size))
    times = np.linspace(0.0, horizon, m_steps + 1)
    n, d = x.shape

    positions = np.empty((m_steps + 1, n, d), dtype=np.float64)
    positions[0] = x
    drifts = np.empty((m_steps, n, d), dtype=np.float64) if store_drifts else None

    solver = ScalarTorusHeatSemigroup(observable, dimension=d, grid_shape=grid_shape)
    noise_scale = np.sqrt(2.0 * step_size / masses)[:, None]

    for m in range(m_steps):
        tau = horizon - times[m]
        drift = np.empty((n, d), dtype=np.float64)

        for i in range(n):
            value, gradient = solver.evaluate(c * masses[i], tau / masses[i], x[i])
            drift[i] = (2.0 / masses[i]) * gradient / max(value, 1e-14)

        x = wrap_torus(x + drift * step_size + noise_scale * rng.normal(size=(n, d)))
        positions[m + 1] = x

        if drifts is not None:
            drifts[m] = drift

    return ParticleSimulation(times=times, positions=positions, masses=masses, drifts=drifts)


def radial_statistics(positions, *, center=(0.5, 0.5), target_radius=0.16, ring_sigma=0.03, hole_sigma=0.05):
    disp = shortest_periodic_displacement(positions, np.asarray(center, dtype=np.float64))
    r = np.sqrt(np.sum(disp ** 2, axis=-1))
    ring_score = np.mean(np.exp(-0.5 * ((r - target_radius) / ring_sigma) ** 2), axis=-1)
    hole_score = np.mean(np.exp(-0.5 * (r / hole_sigma) ** 2), axis=-1)
    mean_radius = np.mean(r, axis=-1)
    return ring_score, hole_score, mean_radius


In [ ]:

# --- Main parameters for the exact cylinder annulus example ---

n_particles = 192
masses = np.full(n_particles, 1.0 / n_particles)

center = np.array([0.5, 0.5], dtype=np.float64)
initial_radius = 0.055

target_radius = 0.16
target_sigma = 0.045
conditioning_strength = 500.0

horizon = 2.0e-5
n_steps = 80
step_size = horizon / n_steps

grid_shape = 32
seed = 0

observable = periodic_annulus_observable(
    center=center,
    radius=target_radius,
    sigma=target_sigma,
    image_radius=1,
)

initial_positions = sunflower_disk(
    n_particles,
    center=center,
    radius=initial_radius,
)

disp0 = shortest_periodic_displacement(initial_positions, center)
initial_angle = (np.arctan2(disp0[:, 1], disp0[:, 0]) + np.pi) / (2.0 * np.pi)
hover_text = [
    f"particle {i}<br>initial angle = {angle:.3f}"
    for i, angle in enumerate(initial_angle)
]

print("n_particles =", n_particles)
print("conditioning_strength =", conditioning_strength)
print("horizon =", horizon, "step_size =", step_size, "seed =", seed)


In [ ]:

rng_free = np.random.default_rng(seed)
rng_conditioned = np.random.default_rng(seed)

free_sim = simulate_free_diffusion(
    masses=masses,
    horizon=horizon,
    step_size=step_size,
    initial_positions=initial_positions,
    rng=rng_free,
)

conditioned_sim = simulate_exact_cylinder_em(
    masses=masses,
    observable=observable,
    c=conditioning_strength,
    horizon=horizon,
    step_size=step_size,
    initial_positions=initial_positions,
    grid_shape=grid_shape,
    rng=rng_conditioned,
    store_drifts=True,
)

print("free positions shape:", free_sim.positions.shape)
print("conditioned positions shape:", conditioned_sim.positions.shape)
print("mean first-step drift norm:", float(np.mean(np.linalg.norm(conditioned_sim.drifts[0], axis=1))))


In [ ]:

static_traces_left = [
    circle_trace(center, target_radius, name="target annulus", color="rgba(220,20,60,0.60)", dash="dash"),
]

static_traces_right = [
    circle_trace(center, target_radius, name="target annulus", color="rgba(220,20,60,0.60)", dash="dash"),
]

comparison_fig = make_comparison_animation(
    free_positions=free_sim.positions,
    conditioned_positions=conditioned_sim.positions,
    times=free_sim.times,
    color_values=initial_angle,
    hover_text=hover_text,
    static_traces_left=static_traces_left,
    static_traces_right=static_traces_right,
    title="Exact cylinder h-transform: central blob → annulus",
    marker_size=8,
)

comparison_fig.show()


In [ ]:

free_ring_score, free_hole_score, free_mean_radius = radial_statistics(
    free_sim.positions,
    center=center,
    target_radius=target_radius,
)

cond_ring_score, cond_hole_score, cond_mean_radius = radial_statistics(
    conditioned_sim.positions,
    center=center,
    target_radius=target_radius,
)

score_fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Annulus score (higher is better)", "Central-hole score (lower is better)"),
    horizontal_spacing=0.12,
)

score_fig.add_trace(
    go.Scatter(x=free_sim.times, y=free_ring_score, mode="lines", name="free annulus score"),
    row=1,
    col=1,
)
score_fig.add_trace(
    go.Scatter(x=conditioned_sim.times, y=cond_ring_score, mode="lines", name="conditioned annulus score"),
    row=1,
    col=1,
)

score_fig.add_trace(
    go.Scatter(x=free_sim.times, y=free_hole_score, mode="lines", name="free center score", line=dict(dash="dash")),
    row=1,
    col=2,
)
score_fig.add_trace(
    go.Scatter(x=conditioned_sim.times, y=cond_hole_score, mode="lines", name="conditioned center score", line=dict(dash="dash")),
    row=1,
    col=2,
)

score_fig.update_layout(
    title="The conditioned system fills the target annulus while opening a hole at the centre",
    template="simple_white",
    width=980,
    height=420,
)
score_fig.update_xaxes(title_text="time", row=1, col=1)
score_fig.update_xaxes(title_text="time", row=1, col=2)
score_fig.update_yaxes(title_text="score", row=1, col=1)
score_fig.update_yaxes(title_text="score", row=1, col=2)
score_fig.show()

print("final annulus score  | free =", float(free_ring_score[-1]), "| conditioned =", float(cond_ring_score[-1]))
print("final centre score   | free =", float(free_hole_score[-1]), "| conditioned =", float(cond_hole_score[-1]))
print("final mean radius    | free =", float(free_mean_radius[-1]), "| conditioned =", float(cond_mean_radius[-1]))


In [ ]:

# --- Seed sweep: quantify stability across seeds and keep a few final states for visual inspection ---

seed_list = list(range(8))
final_free_by_seed = {}
final_conditioned_by_seed = {}
records = []

for sweep_seed in seed_list:
    free = simulate_free_diffusion(
        masses=masses,
        horizon=horizon,
        step_size=step_size,
        initial_positions=initial_positions,
        rng=np.random.default_rng(sweep_seed),
    )

    conditioned = simulate_exact_cylinder_em(
        masses=masses,
        observable=observable,
        c=conditioning_strength,
        horizon=horizon,
        step_size=step_size,
        initial_positions=initial_positions,
        grid_shape=grid_shape,
        rng=np.random.default_rng(sweep_seed),
        store_drifts=False,
    )

    free_ring, free_hole, free_radius = radial_statistics(
        free.positions[-1][None, :, :],
        center=center,
        target_radius=target_radius,
    )
    cond_ring, cond_hole, cond_radius = radial_statistics(
        conditioned.positions[-1][None, :, :],
        center=center,
        target_radius=target_radius,
    )

    records.append(
        {
            "seed": sweep_seed,
            "free_ring_score": float(free_ring[0]),
            "conditioned_ring_score": float(cond_ring[0]),
            "free_centre_score": float(free_hole[0]),
            "conditioned_centre_score": float(cond_hole[0]),
            "free_mean_radius": float(free_radius[0]),
            "conditioned_mean_radius": float(cond_radius[0]),
        }
    )

    if sweep_seed < 4:
        final_free_by_seed[sweep_seed] = free.positions[-1]
        final_conditioned_by_seed[sweep_seed] = conditioned.positions[-1]

scores_df = pd.DataFrame(records)
scores_df["ring_score_gain"] = scores_df["conditioned_ring_score"] - scores_df["free_ring_score"]
scores_df["centre_score_drop"] = scores_df["free_centre_score"] - scores_df["conditioned_centre_score"]

scores_df.round(4)


In [ ]:

seed_fig = make_seed_grid_figure(
    final_free_by_seed=final_free_by_seed,
    final_conditioned_by_seed=final_conditioned_by_seed,
    color_values=initial_angle,
    center=center,
    target_radius=target_radius,
)
seed_fig.show()


In [ ]:

stability_fig = go.Figure()

stability_fig.add_trace(
    go.Scatter(
        x=scores_df["seed"],
        y=scores_df["free_ring_score"],
        mode="lines+markers",
        name="free ring score",
    )
)
stability_fig.add_trace(
    go.Scatter(
        x=scores_df["seed"],
        y=scores_df["conditioned_ring_score"],
        mode="lines+markers",
        name="conditioned ring score",
    )
)
stability_fig.add_trace(
    go.Scatter(
        x=scores_df["seed"],
        y=scores_df["ring_score_gain"],
        mode="lines+markers",
        name="gain",
        line=dict(dash="dash"),
    )
)

stability_fig.update_layout(
    title="Seed sweep: the annulus score stays uniformly better under the exact cylinder h-transform",
    template="simple_white",
    width=900,
    height=420,
    xaxis_title="seed",
    yaxis_title="score",
)
stability_fig.show()

print("minimum ring-score gain over the seed sweep:", float(scores_df["ring_score_gain"].min()))
print("mean ring-score gain over the seed sweep:", float(scores_df["ring_score_gain"].mean()))


## Summary

This example satisfies the practical criteria that were problematic for the nonlinear cylinder examples:

- the difference from the free diffusion is visually obvious
- it remains visible across multiple seeds
- it still works with well over 100 particles
- it uses the paper's exact Doob h-transform framework for a positive cylinder terminal datum

If you want to keep pushing toward Algorithm 2 specifically later, this exact annulus example is also a good baseline for debugging because it separates the h-transform machinery from the extra oscillatory quadrature layer.
